# HCCL 典型算子与算法

上一章我们建立了「通信成员 → 通信域 → 通信算子 → 通信算法」的整体认知：
**算子规定「结果」，算法规定「过程」**。本章就把这两层分别展开——
先认全算子，再看算法如何把算子高效地实现出来。

本章共分三节，建议按顺序学习。

## 本章目录

| 小节 | 主题 | 打开 |
| --- | --- | --- |
| **2.1 本章介绍** | 学习目标、全章目录、术语与缩略语 | 本文件 |
| **2.2 HCCL 算子介绍** | 8 个常见集合通信算子 + 大模型的典型通信模型 | [2.2 HCCL 算子介绍](./02.02_operators.ipynb) |
| **2.3 HCCL 算法简介** | 为什么需要算法 / Ring 算法（含动图）/ Mesh 与 Halving-Doubling | [2.3 HCCL 算法简介](./02.03_algorithms.ipynb) |

---

本章课程请从 [HCCL 算子介绍](./02.02_operators.ipynb) 开始学习。

### 2.2 HCCL 算子介绍（先认全「算子」）

1. **常见集合通信算子总览** —— 按「多对一 / 一对多」与「多对多」两大类，逐个认识 8 个算子；
2. **多对一与一对多** —— 四个带 root 的算子：Gather / Reduce / Broadcast / Scatter；
3. **多对多** —— 四个无 root 的算子：AllReduce / AllGather / ReduceScatter / AlltoAll；
4. **大模型的典型通信模型** —— TP / PP / DP / EP 四类并行分别使用哪些算子、
   数据量多大、能否被计算隐藏。

### 2.3 HCCL 算法简介（再看「算法」）

1. **为什么需要集合通信算法** —— 上万张卡同时通信会怎样？「统一流量规划」解决什么问题；
2. **Ring 算法** —— 数据沿环逐跳传递，N-1 步完成（配有**动画演示**）；
3. **Mesh 与 Halving-Doubling 算法** —— 一步全连通 vs. log₂N 步、步长逐级翻倍。

## 学习目标

学完本章，你应当能够：

- 说出 8 个常见集合通信算子的语义，并判断其属于哪一类别；
- 结合四类并行方式，说出各自对应的通信算子与典型数据量；
- 解释「为什么集合通信需要算法」——流量规划与拓扑适配；
- 描述 Ring 算法的逐跳过程，并说出其通信步数；
- 区分 Mesh 与 Halving-Doubling 的适用拓扑与步数规律。

## 术语与缩略语

| 概念 | 说明 |
| --- | --- |
| Rank | 参与通信的最小逻辑实体（一颗芯片、一个 Die 或一个进程） |
| 通信域 | 一组通信成员的组合，描述通信范围；一个集群可建多个域 |
| 通信算子 | 在通信域内完成通信任务的算子；域内算子**串行执行** |
| 通信算法 | 完成某个算子的**具体实现方案**，决定链路选择与分步方式 |
| root | 「一对多 / 多对一」算子中的**中心成员**，负责汇聚或分发数据 |
| 点到点通信 | Send / Recv：仅两个成员之间的通信，不属于集合通信 |
| All2All | 通信域内所有成员相互 scatter 与 gather 等量数据的算子 |
| 流量规划 | 对通信数据在物理链路与时间上的统一编排，用以避免拥塞、提升带宽利用率 |